In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

# Initialize LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.1-8b-instant",
    temperature=0.1
)

print("LangChain + Groq initialized")
print("Day 11 - LangChain")

# Quick test
response = llm.invoke("Say 'LangChain is ready' and nothing else.")
print(f"\nTest: {response.content}")


LangChain + Groq initialized
Day 11 - LangChain

Test: LangChain is ready.


In [3]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

print("=== PromptTemplate ===\n")

# Basic PromptTemplate - reusable prompt with variables
rag_prompt = PromptTemplate(
    input_variables =  ["context", "question"],
    template="""You are an Enterprise RAG assistant.
Answer the question based only on the provided documents.
If the answer is not in the documents say 'I cannot find this.'

Documents:
{context}

Question: {question}

Answer:"""
)

# Format the prompt with actual values
formatted = rag_prompt.format(
    context = "RAG uses hybrid search combining BM25 and combining BM25 and vector search.",
    question = "What search methods does RAG use?"
)

print("Formatted prompt:")
print(formatted)
print(f"\nInput variables: {rag_prompt.input_variables}")

# ChatPromptTemplate - for chat models
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "you are an Enterprise RAG assistant. Answer only from provided context."),
    ("human", "Documentd:\n{context}\nQuestion: {question}")
])

formatted_chat = chat_prompt.format_messages(
    context="Hybrid search merges BM25 and vector results using RRF.",
    question="How are search results merged?"
)

print(f"\nChat messages:")
for msg in formatted_chat:
    print(f"  [{msg.__class__.__name__}]: {msg.content[:80]}...")

=== PromptTemplate ===

Formatted prompt:
You are an Enterprise RAG assistant.
Answer the question based only on the provided documents.
If the answer is not in the documents say 'I cannot find this.'

Documents:
RAG uses hybrid search combining BM25 and combining BM25 and vector search.

Question: What search methods does RAG use?

Answer:

Input variables: ['context', 'question']

Chat messages:
  [SystemMessage]: you are an Enterprise RAG assistant. Answer only from provided context....
  [HumanMessage]: Documentd:
Hybrid search merges BM25 and vector results using RRF.
Question: How...


In [4]:
from langchain_core.output_parsers import StrOutputParser

print("=== LangChain Chains ===\n")

# The pip operator | Chain components together
# prompt | llm | parser
# This is called LCEL - LangChain Expression Language

# Build a simple RAG chain
rag_chain = chat_prompt | llm | (lambda x: x.content)

# Invoke the Chain
response = rag_chain.invoke({
    "context": """Hybrid search combines BM25 keyword search with vector
semantic search. Results are merged using Reciprocal Rank Fusion(RRF).
Re-ranking uses a cross-encoder to improve precision after retrieval.""",
    "question": "What is the difference between hybrid search and re-ranking?"
})

print(f"Chain response:\n{response}")
print(f"\nType: {type(response)}") # plain string not a message object

# Chain with differnt prompt
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a document summarizer. Be concise."),
    ("human", "summarize this in exactly 2 sentences:\n{text}")
])

summary_chain = summary_prompt | llm | StrOutputParser()

summary = summary_chain.invoke({
    "text": """Enterprise RAG pipeline combine multiplr retrieval strategies
including BM25 keyword search and vector semantic search. The results
are merged using Reciprocal Rank Fusion and then re-ranked using a 
cross-encoder model. RAGAs framework evaluates the pipeline using
faithfulness and relevancy metrics to detect hallucinations."""
})

print(f"\nSummary:\n{summary}")

=== LangChain Chains ===

Chain response:
Based on the provided context, the difference between hybrid search and re-ranking is as follows:

- **Hybrid Search**: This combines two search methods:
  1. **BM25 Keyword Search**: A traditional search method that uses keyword matching to retrieve relevant documents.
  2. **Vector Semantic Search**: A search method that uses vector representations of documents to retrieve relevant documents based on semantic similarity.
  The results from both methods are then merged using **Reciprocal Rank Fusion (RRF)** to produce the final search results.

- **Re-ranking**: This is a process that takes the initial search results and uses a **cross-encoder** to re-rank them based on their relevance. The goal of re-ranking is to improve the precision of the search results by re-ordering them based on their relevance to the query.

Type: <class 'str'>

Summary:
The Enterprise RAG pipeline combines multiple retrieval strategies, including BM25 keyword search 

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

print("=== Document Loader & Text splitter ===\n")

# Create a simple document to load
sample_doc = """# Enterprise RAG Pipeline Documentation

## Overview
Enterprise RAG (Retrieval Augmented Generation) is a system that combines 
document retrieval with language model generation. Unlike basic RAG systems,
enterprise versions include hybrid search, re-ranking, and evaluation.

## Hybrid Search
Hybrid search combines two retrieval methods:
1. BM25 keyword search - finds exact keyword matches
2. Vector semantic search - finds semantically similar content
Results are merged using Reciprocal Rank Fusion (RRF).

## Re-ranking
After hybrid retrieval, a cross-encoder model re-scores the top chunks.
Cross-encoders process query and document together for higher precision.
This step significantly improves retrieval accuracy.

## Evaluation with RAGAs
RAGAs framework measures pipeline quality using four metrics:
- Faithfulness: is the answer grounded in retrieved context?
- Answer Relevancy: does the answer address the question?
- Context Recall: were all relevant chunks retrieved?
- Context Precision: were irrelevant chunks filtered out?

## Multi-tenancy
Each organization gets an isolated ChromaDB namespace.
User authentication via JWT tokens controls document access.
Company A cannot access Company B's documents.

## Deployment
The pipeline runs on FastAPI with Docker containerization.
Deployed on AWS EC2 with Nginx as reverse proxy.
GitHub Actions handles CI/CD automation.
"""

# Save the file
with open("rag_documentation.txt", "w") as f:
    f.write(sample_doc)

# Load using TextLoader
loader = TextLoader("rag_documentation.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document")
print(f"content length: {len(documents[0].page_content)} characters")
print(f"Metadata: {documents[0].metadata}")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)

print(f"\nSplit into {len(chunks)} chunks")
print(f"\nChunk sizes:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk{i+1}: {len(chunk.page_content)} chars - {chunk.page_content[:60]}...")

=== Document Loader & Text splitter ===

Loaded 1 document
content length: 1388 characters
Metadata: {'source': 'rag_documentation.txt'}

Split into 7 chunks

Chunk sizes:
  Chunk1: 275 chars - # Enterprise RAG Pipeline Documentation

## Overview
Enterpr...
  Chunk2: 233 chars - ## Hybrid Search
Hybrid search combines two retrieval method...
  Chunk3: 211 chars - ## Re-ranking
After hybrid retrieval, a cross-encoder model ...
  Chunk4: 259 chars - ## Evaluation with RAGAs
RAGAs framework measures pipeline q...
  Chunk5: 57 chars - - Context Precision: were irrelevant chunks filtered out?...
  Chunk6: 179 chars - ## Multi-tenancy
Each organization gets an isolated ChromaDB...
  Chunk7: 162 chars - ## Deployment
The pipeline runs on FastAPI with Docker conta...


In [6]:
print("=== Chunking Strategy Comparison ===\n")

test_text = documents[0].page_content

# Strategy 1 - Fixed size (naive)
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0 # no overlap
)
fixed_chunks = fixed_splitter.split_text(test_text)

# Strategy 2 - Recursive with overlap (recommended)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=40,
    separators=["\n\n", "\n", ".", " "]
)
recursive_chunks = recursive_splitter.split_text(test_text)


# Strategy 3 - Large chunks (for dense documents)
large_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100 
)
large_chunks = large_splitter.split_text(test_text)

print(f"Strategy 1 - Fixed (no overlap):")
print(f"  Chunks: {len(fixed_chunks)}")
print(f"  Avg size: {sum(len(c) for c in fixed_chunks)//len(fixed_chunks)} chars")

print(f"Strategy 2 - Recursive with overlap:")
print(f"  Chunks: {len(recursive_chunks)}")
print(f"  Avg size: {sum(len(c) for c in recursive_chunks)//len(recursive_chunks)} chars")

print(f"Strategy 3 - Fixed (no overlap):")
print(f"  Chunks: {len(large_chunks)}")
print(f"  Avg size: {sum(len(c) for c in large_chunks)//len(large_chunks)} chars")

# Show overlap in action
print(f"\n=== Overlap Demonstration ===")
print(f"End of chunk 1:\n  ...{recursive_chunks[0][-60:]}")
print(f"\nStart of chunks 2:\n {recursive_chunks[1][:60]}...")
print(f"\nOverlapping text preserved across chunk boundary ✅")


=== Chunking Strategy Comparison ===

Strategy 1 - Fixed (no overlap):
  Chunks: 11
  Avg size: 124 chars
Strategy 2 - Recursive with overlap:
  Chunks: 11
  Avg size: 124 chars
Strategy 3 - Fixed (no overlap):
  Chunks: 4
  Avg size: 345 chars

=== Overlap Demonstration ===
End of chunk 1:
  ...# Enterprise RAG Pipeline Documentation

Start of chunks 2:
 ## Overview
Enterprise RAG (Retrieval Augmented Generation) ...

Overlapping text preserved across chunk boundary ✅


In [9]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("=== Complete RAG Chain===\n")
print("Setting up embrddings and vector store...\n")

# Embeddings - same model from Day 8
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"}
)

# Create vector store from chunks
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings,
    collection_name = "rag_docs"
)

print(f"Vector store created with {len(chunks)} chunks")

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} # return top 3 chunks
)

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an Enterprise RAG assistant.
Answer question based only on the provided context.
If the answer is not in the context say 'I cannot find this in the documents.'
Always be concise and cite the relevant information."""),
    ("human", "context:\n{context}\n\nQuestion: {question}")
])

# Helper of format retrieved docs
def format_docs(docs):
    return "\n\n".join([
        f"[Chunk {i+1}]: {doc.page_content}"
        for i, doc in enumerate(docs)
    ])

# Build complete RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | (lambda x: x.content)
)

# Test queries
queries = [
    "How does hybrid search work?",
    "What metrics does RAGAs use?",
    "How is multi-tenancy implemented?"
]

print("=== Testing RAG Chain ===\n")
for query in queries:
    print(f"Q: {query}")
    response = rag_chain.invoke(query)
    print(f"A: {response}\n")
    print("-" * 50 + "\n")

=== Complete RAG Chain===

Setting up embrddings and vector store...

Vector store created with 7 chunks
=== Testing RAG Chain ===

Q: How does hybrid search work?
A: Hybrid search combines two retrieval methods:
1. BM25 keyword search - finds exact keyword matches
2. Vector semantic search - finds semantically similar content
Results are merged using Reciprocal Rank Fusion (RRF).

--------------------------------------------------

Q: What metrics does RAGAs use?
A: RAGAs uses four metrics: 
- Faithfulness
- Answer Relevancy
- Context Recall

--------------------------------------------------

Q: How is multi-tenancy implemented?
A: Multi-tenancy is implemented through two methods: 

1. Isolated ChromaDB namespace for each organization (Chunk 1, Chunk 2, Chunk 3).
2. User authentication via JWT tokens to control document access (Chunk 1, Chunk 2, Chunk 3).

--------------------------------------------------

